In [ ]:
import os 
from dotenv import load_dotenv
load_dotenv(override=True)
from langchain_community.document_loaders import PyPDFLoader
PDF_PATH = "/Volumes/Work/HAC_Langchain_Demo/langchain_101/gpt-1paper.pdf"

# create the loader object
loader = PyPDFLoader(PDF_PATH)

# create the document object
docs = loader.load()

print(f"Number of document objects created from the pdf : {len(docs)}")
print(f"Page content of the first document object : {docs[0].page_content}")

In [ ]:
# inspect pages

# check document object : what is the length , preview it 


for i,doc in enumerate(docs):
    preview = doc.page_content[:100].replace("\n"," ")
    print(f"{i} | {len(doc.page_content)} | {preview} ...")

In [ ]:
# what is enumerate
a = [1,2,3,4,5,6]

for i, val in enumerate(a):
    print(i, val)

In [ ]:
# text splitting

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 200,
    chunk_overlap = 80
)

chunks = text_splitter.split_documents(docs)
print(f"Chunks created : {len(chunks)}")
print(f"Page content of the first chunk with 100 characters : {chunks[0].page_content[:100]}")

In [ ]:
# inspect chunks
# loop over 5 chunks

for i in range(5):
    print(f"Chunk : {i}")
    print(f"Metadata : {chunks[i].metadata}")
    print(f"Length of chunk : {len(chunks[i].page_content)}")
    print(f"Content of chunk : {chunks[i].page_content[:100]} ...")
    print("--------------------------------------------------")

In [ ]:
# embeddings
from langchain_openai import OpenAIEmbeddings
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

# take a sample chunk and check if embedding works on it 
sample_chunk = chunks[0].page_content
sample_vector = embedding_model.embed_query(sample_chunk)
print(f"Sample chunk : {sample_chunk[:100]} ...")
print(f"Length of the embedding vector : {len(sample_vector)}")
print(f"First 10 dimensions of the embedding vector : {sample_vector[:10]}")

In [ ]:
# store my vectors in the vector database

from langchain_chroma import Chroma
print(f"Embedding {len(chunks)} chunks into the vector database.......")

vectorstore = Chroma.from_documents(
    documents= chunks,
    embedding= embedding_model,
    persist_directory= "./hac_rag_db",
    collection_name = "gpt_1_paper"
)

total = vectorstore._collection.count()

In [ ]:
print("Total chunks pushed in the vector database : ", total)

In [ ]:
# peek into vector database

collection_data = vectorstore._collection.get(include=["metadatas", "documents", "embeddings"])

ids = collection_data['ids']
embeddings = collection_data['embeddings']
metadatas = collection_data['metadatas']
documents = collection_data['documents']


for i in range(5):
    print(f"Chunk ID: {ids[i]}")
    print(f"Metadata: {metadatas[i]}")
    print(f"Chunk content : {documents[i][:100]}")  # Print first 100 characters of the document
    print(f"Embedding length: {len(embeddings[i])}")
    print(f"First 10 dimensions of the embedding: {embeddings[i][:10]}")
    print("---------a-----------------------------------------")

In [ ]:
def retrieve_with_scores(query:str,k:int=3):
    "retrieve the top k chunks with cosine scores"
    results = vectorstore.similarity_search_with_relevance_scores(query=query, k=k)
    print(f"Query : {query}")
    print(f"Top {k} results : ")
    for i,(doc,score) in enumerate(results):
        print(f"Rank {i+1} | score: {score : .4f} | Page : {doc.metadata.get('page', '?')}")
        print(f"{doc.page_content[:200]}")  # Print first 200 characters of the document
        print()
    return results

In [ ]:
result = retrieve_with_scores("What datasets were used in Natural Language Inference? ")
print(result)